<a href="https://colab.research.google.com/github/tevfikaytekin/reccommender_systems_course/blob/main/knn_cf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY_NAME/blob/main/YOUR_NOTEBOOK_NAME.ipynb)

<!-- Update the URL above with your actual GitHub username, repository, and notebook file name -->

# Neighborhood-based Collaborative Filtering
(by Tevfik Aytekin)

In the following we will implement item based collaborative filtering.

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from scipy.sparse import csr_matrix
from collections import Counter
from sklearn.metrics import pairwise_distances
from operator import itemgetter
from tqdm.notebook import tqdm
import copy
import heapq
import sys, os
import pickle
import itertools
import operator


In [7]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o ml-latest-small.zip

--2026-09-05 09:53:44--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
  Issued certificate has expired.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  4.46MB/s    in 0.2s    

2026-09-05 09:53:44 (4.46 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


### Movielens ml-latest-small dataset

In [8]:
with open('ml-latest-small/README.txt', 'r') as f:
    print(f.read())

Summary

This dataset (ml-latest-small) describes 5-star rating and free-text tagging activity from [MovieLens](http://movielens.org), a movie recommendation service. It contains 100836 ratings and 3683 tag applications across 9742 movies. These data were created by 610 users between March 29, 1996 and September 24, 2018. This dataset was generated on September 26, 2018.

Users were selected at random for inclusion. All selected users had rated at least 20 movies. No demographic information is included. Each user is represented by an id, and no other information is provided.

The data are contained in the files `links.csv`, `movies.csv`, `ratings.csv` and `tags.csv`. More details about the contents and use of all these files follows.

This is a *development* dataset. As such, it may change over time and is not an appropriate dataset for shared research results. See available *benchmark* datasets if that is your intent.

This and other GroupLens data sets are publicly available for down

In [11]:
ratings = pd.read_csv("ml-latest-small/ratings.csv", sep=",")
print(ratings.shape)
ratings.head(10)

(100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


In [12]:
links = pd.read_csv("ml-latest-small/links.csv", sep=",")
print(links.shape)
links.head()

(9742, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [13]:
movies = pd.read_csv("ml-latest-small/movies.csv", sep=",")
print(movies.shape)
movies.head()

(9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [14]:
tags = pd.read_csv("ml-latest-small/tags.csv", sep=",")
print(tags.shape)
tags.head()

(3683, 4)


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


## Create User Item Rating Map
It might take some time but will be useful later.

In [ ]:
rating_map = {}
for i in range(len(ratings)):
    key = str(ratings.iloc[i,0]) + '_' +str(ratings.iloc[i,1])
    rating_map[key]=ratings.iloc[i,2]

In [ ]:
rating_map["1_101"]

In [ ]:
iterator = iter(rating_map.items())
for i in range(5):
    print(next(iterator))

## Create User Ratings Map
It might take some time but will be useful later.

In [ ]:
ratings.query("movieId == 2")

In [ ]:
# user_ratings_map[i] stores a tuple: a list of users who rated item i and a list of corresponding ratings
user_ratings_map = {}

items = ratings.movieId.unique()
for i in items:
    userids = ratings.query("movieId == @i").userId.array
    user_ratings = ratings.query("movieId == @i").rating.array
    user_ratings_map[i] = (userids,user_ratings)

In [ ]:
user_ratings_map[10]

## Rating Prediction

### Algorithm

Predict rating of user $u$ for item $i$
- Calculate the similarity of items that are rated by $u$ with $i$.
- Use these similarities to calculate a weighted average of the ratings.

Similarity between items i and j will be calculated using the ratings of i and j (no content information will be used). One can view these ratings as a vector of values as shown below and use different metrics such as Jaccard or cosine.

In [ ]:
df = pd.DataFrame([[1, "", 5, 3, ""],
                   [5, 3, "", 2, 4],
                   ["", 4, 2, "", 1],
                   [3, "", 4, 2, 3],
                   [4, 1, "", 2, 4],
                   [4, 1, 5, 3, ""],
                   ["", 4, 5, "", 1],
                   [2, 5, "", 1, 4]],
                 index = ['user 1','user 2','user 3','user 4','user 5','user 6','user 7','user 8'],
                 columns = ['movie 1','movie 2','movie 3','movie 4',' movie 5'])
df

NameError: name 'pd' is not defined

### Jaccard Similarity

Given two sets $A$ and $B$,

$Jaccard(A, B) = \frac{|A \cap B|}{|A \cup B|}$

For example if $A = \{a, b, c, d\}$ and $B = \{b, d, e ,f, g\}$ then

$Jaccard(A, B) = \frac{2}{7}$

We can apply Jaccard similarity by ignoring the rating values.

### Cosine Similarity

Cosine(A, B) =

<img src="cosine.png" width="200">



In [ ]:
def NNCF_based_rating_prediction(u, i, metric):
    r = 0
    sum_sim = 0
    # find movies rated by u
    movies = ratings[ratings["userId"]==u].movieId
    for j in movies:
        sim = calc_sim(i, j, metric)
        key = str(u)+"_"+str(j)
        r += sim*rating_map[key]
        sum_sim += sim
    if sum_sim == 0:
        return 0
    else:
        return r / sum_sim

In [ ]:
# finds the similary of items i and j
def calc_sim(i,j, metric):
    # users who rated item i
    users_rated_i = user_ratings_map[i][0]
    ratings_i = user_ratings_map[i][1]
    # users who rated item j
    users_rated_j = user_ratings_map[j][0]
    ratings_j = user_ratings_map[j][1]

    # Jaccard ignores rating values.
    if metric == "Jaccard":
        intersection_size = len(set(users_rated_i).intersection(users_rated_j))
        union_size = len(set(users_rated_i).union(users_rated_j))
        return intersection_size / union_size
    elif metric == "Cosine":
        inter, ind1, ind2 = np.intersect1d(users_rated_i, users_rated_j, return_indices=True)
        dot = np.dot(ratings_i[ind1], ratings_j[ind2])
        return dot/(np.linalg.norm(ratings_i[ind1])*np.linalg.norm(ratings_j[ind2]))


### Example

In [ ]:
users_i = np.array([1, 3, 6, 9, 12, 15])
ratings_i =  np.array([4, 3, 2, 4, 2, 5])
users_j = np.array([1, 6, 8, 12])
ratings_j = np.array([2, 4, 2, 5])

In [ ]:
inter, ind1, ind2 = np.intersect1d(users_i,users_j,return_indices=True)
print(inter)
print(ind1)
print(ind2)
print(ratings_i[ind1])
print(ratings_j[ind2])
print(np.dot(ratings_i[ind1],ratings_j[ind2]))

## Evaluation of Rating Prediction

How can we measure the performance of a recommender algorithm? This is similar to the evaluation used in machine learning.

- Make a train/test split
- Build the model on the training set
- Make predictions for the ratings in the test set
- Find the mean absolute error (MAE)

For more metrics other then MAE lool at the "Metrics for Regression" section of [this notebook](../../data_science/evaluation.ipynb). For ranking metrics see [this notebook](ranking_evaluation.ipynb)


In [ ]:
ratings = shuffle(ratings)

X_train, X_test = train_test_split(ratings, test_size=100)
train_size = X_train.shape[0]
test_size = X_test.shape[0]
print("Test size:", test_size)
error1 = 0
error2 = 0
error3 = 0
preds = []

avg_rating = X_train.iloc[:,2].mean()
for k in tqdm(range(test_size)):
    u = X_test.iloc[k,0]
    i = X_test.iloc[k,1]
    r = X_test.iloc[k,2]

    error1 += np.abs(r - NNCF_based_rating_prediction(u,i,"Cosine"))
    error2 += np.abs(r - NNCF_based_rating_prediction(u,i,"Jaccard"))
    error3 += np.abs(r - avg_rating)


print("Cosine:", error1/test_size)
print("Jaccard:",error2/test_size)
print("Average:",error3/test_size)


#### Question: How can you explain the difference between using Cosine vs. Jaccard?

## Top-N recommendation Algorithm - Predict and Sort
The task in top-$N$ recommendation is to recommend $N$ items to a user.


Recommend $N$ movies to user $u$
- Predict the ratings of all items which are not watched by $u$
- Sort the predicted ratings
- Recommend the movies with the highest predicted ratings

In [ ]:
def top_N_pred_sort(N, u):
    preds = pd.Series([], dtype='float')
    # find the movies not rated by u
    all_items = set(ratings['movieId'].unique())
    rated_by_user = set(ratings[ratings["userId"]==1].movieId.unique())
    not_rated_by_user = all_items - rated_by_user
    sample_movies = np.random.choice(list(not_rated_by_user), 500)
    for m in tqdm(sample_movies):
        preds[m] = NNCF_based_rating_prediction(u, m, "Jaccard")
    return preds.sort_values(ascending=False)[:N]

In [ ]:
topn = top_N_pred_sort(10, 100)
topn

## Efficiency Issues

There are important inefficiencies in this algorithm:

- The algorithm predicts the rating of all items which are not rated by the user. In the case of millions of items this algorithm is practically infeasible. Numerous techniques have been developed to remedy this problem. Can you suggest a solution?
- In rating prediction, similarity between target item and items rated by the user are calculated. To make a recommendation to another user similarity calculations will be done again. For making recommendations to users in general many similarity calculations will be repeated. A general solution to this problem is to precalculate the similarities between items. Moreover, you don't need to store all similarities, only storing $k$ most similar items to every item will be enough. Size of $k$ can be determined according to the needs.


## Top-N recommendation Algorithm - kNN Map
The task in top-$N$ recommendation is to recommend $N$ items to a user.

- Build a knn-map (a map which stores the $k$ nearest neighbors of each item)

Recommend $N$ movies to user $u$
- Get the neigbors of movies which are watched by $u$ and put them into a list $C$.
- Choose $N$ movies form $C$. There can be different methods here. Most repeated movies in C can be chosen, movies with the highest total similarity (or maximum similarity) can be chosen. These methods will be implemented.
- Recommend the $N$ movies that are chosen.

## Building a knn map
This table will hold the most similar $k$ items for each item. In order to build this table we need to calculate all pairwise similarities which takes $O(n^2)$ time. There is no escape from this $O(n^2)$ time unless you use an approximation algorithm such as LSH (Locality Sensitive Hashing) for nearest neighbor search. Another approach might be to calculate similarites by a matrix multiplication operation and give it to a GPU for accelaration. Yet another approach is discovered by Amazon researchers which can lead to huge speed ups for very sparse matrices which we will look at below.

We will use a heap based priority queue for storing the nearest neighbor. You can look at this [animation](https://www.cs.usfca.edu/~galles/visualization/Heap.html).

In [ ]:
pq =[(10,"a"),(8, "b"), (5, "c"), (3, "d")]
type(pq)
heapq.heapify(pq)
heapq.nsmallest(2,pq)

In [ ]:
movies[:10]

In [ ]:
def build_knn_map(movies, K=30):
    knn_map = {}
    movie_ids = ratings['movieId'].unique()
    print(len(movie_ids))
    for i in tqdm(movie_ids):
        pq = []
        knn_map[i] = pq
        for j in movie_ids:
            if (i == j):
                continue
            sim = calc_sim(i,j,"Jaccard")
            if (len(pq) >= K):
                smallest = pq[0]
                if (sim > smallest[0]):
                    heapq.heappop(pq)
                    heapq.heappush(pq, (sim, j))
            else:
                heapq.heappush(pq, (sim, j))
    return knn_map

In [ ]:
knn_map = build_knn_map(movies)

### Amazon Algorithm

The following is the algorithm used at Amazon. Note that this does not help much for the movielens dataset, however, for very sparse datasets such as the dataset at Amazon, it really helps by skipping many item pairs for which there is no user which rated/bought both items.

Linden, Greg, Brent Smith, and Jeremy York. "Amazon. com recommendations: Item-to-item collaborative filtering." IEEE Internet computing 7.1 (2003): 76-80.


In [ ]:
def build_knn_map_amazon(movies, K=30):
    knn_map = {}
    movie_ids = ratings['movieId'].unique()
    print(len(movie_ids))
    for i in tqdm(movie_ids):
        pq = []
        knn_map[i] = pq
        # find users who rated i
        users = ratings.query("movieId == @i").userId.unique()
        # find items rated by users_i
        movies = ratings.query("userId in @users").movieId.unique()
        # For speed up (this does not exist in the Amazon algorithm)
        movies = np.random.choice(movies, 500)
        for j in movies:
            if (i == j):
                continue
            sim = calc_sim(i,j,"Jaccard")
            if (len(pq) >= K):
                smallest = pq[0]
                if (sim > smallest[0]):
                    heapq.heappop(pq)
                    heapq.heappush(pq, (sim, j))
            else:
                heapq.heappush(pq, (sim, j))
    return knn_map, movies

In [ ]:
knn_map, movies = build_knn_map_amazon(movies)

In [ ]:
def add_sims_and_sort(l):
    li = []
    it = itertools.groupby(l, operator.itemgetter(1))
    for key, subiter in it:
        li.append((key, sum(item[0] for item in subiter)))
    li = sorted(li, key=itemgetter(1), reverse=True)
    return li


In [ ]:
def top_N_knn_map(ratings, N, u):
    C = []
    # find the movies rated by u
    movies_rated = ratings.query("userId == @u").movieId
    for m in movies_rated:
        C = C + knn_map[m]
    return add_sims_and_sort(C)[:N]

In [ ]:
topn = top_N_knn_map(ratings, 10, 100)
topn

In [ ]:
topn = [i[0] for i in topn]
movies[movies.movieId.isin(topn)]

## Evaluation of top-N recommendation

Evaluation of rating prediction is rather easy: find the mean absolute error between rating predictions and true ratings. How can we evaluate the accuracy of a top-N recommendation? There are several techniques which we will look at in more detail later. Below is one common way to evaluate top-N recommendation:

- Randomly sub-sample some portion of positive preferences in order to create a test set $T$. Positive preferences might be 5-star ratings, movies watched more than a certain threshold, or items purchased.
- Put the rest of the preferences into the training set and build model.

- For each preference $(u,i)$ in the test set:
    - Make a top-N recommendation tu user $u$.
    - If the test item i occurs among the top-N items, then we have a hit, otherwise we have a miss.

Hit ratio is then defined as:

$$
Hit Ratio: \frac{\#hits}{|T|}
$$



In [ ]:
N = 100
X_train, X_test = train_test_split(ratings, test_size=1000)
X_test = X_test.query("rating > 4")
train_size = X_train.shape[0]
test_size = X_test.shape[0]
print("Test size:", test_size)
hit_count = 0
for k in range(test_size):
    u = X_test.iloc[k,0]
    i = X_test.iloc[k,1]
    r = X_test.iloc[k,2]
    top_N = top_N_knn_map(X_train, N, u)
    hit_list = [item for item in top_N if item[0] == i]
    if len(hit_list) > 0:
        hit_count +=1
print("Hit Ratio", hit_count/test_size)

### OPTIONAL

The following is a comparison of the running times of similarity calculations performed on the CPU and GPU.

https://github.com/erogluegemen/Performance-and-Scalability-Analysis-of-KNN-Implementations-for-Content-Based-Filtering

## Exercises

**Q1.** How does Jaccard similarity differ from Cosine similarity in the context of collaborative filtering, specifically regarding how they handle the actual values of user ratings? Which one ignores the magnitude of ratings?

**Answer:**
**Jaccard similarity** treats ratings as binary (rated or not rated), considering only the sets of users who interacted with an item. It ignores the actual rating values (e.g., 1 star vs 5 stars). **Cosine similarity** treats the ratings as vectors in a multi-dimensional space, taking into account the magnitude of the ratings (the values themselves) and the angle between the vectors.

---

**Q2.** What is the time complexity (Big O notation) for building a full kNN map (calculating all pairwise similarities) for a dataset with $N$ items? Why might this be considered an inefficiency for large-scale recommender systems?

**Answer:**
Building a full kNN map requires calculating the similarity between every pair of items. For $N$ items, this results in **$O(N^2)$** complexity. This quadratic growth makes it computationally expensive and often infeasible for large-scale systems with millions of items.

---

**Q3.** Explain the fundamental difference between Mean Absolute Error (MAE) and Hit Ratio as evaluation metrics. In what specific recommendation scenario (e.g., Rating Prediction vs. Top-N Recommendation) is each metric most appropriate?

**Answer:**
**Mean Absolute Error (MAE)** measures the average magnitude of errors in predicted ratings (regression task), appropriate when you want to know how close the predicted rating is to the actual rating. **Hit Ratio** measures the effectiveness of a top-N recommendation list (ranking task), determining if the relevant item appears in the recommended set. Hit Ratio is more relevant for scenarios where the goal is to present a list of items the user is likely to interact with.

**Q4.** Solve the following tasks using the **User-Item Rating Matrix** given below:

|  | Item A | Item B | Item C | Item D |
| :--- | :---: | :---: | :---: | :---: |
| **User 1** | 5 | 3 | ? | 1 |
| **User 2** | 4 | ? | 4 | 2 |
| **User 3** | 1 | 1 | ? | 5 |
| **User 4** | 4 | 2 | 3 | ? |
---

**Task 1 (Jaccard):**
Calculate the Jaccard similarity between **Item A** and **Item D**. Recall that Jaccard similarity considers the sets of users who rated each item.

**Solution:**
1.  Identify users who rated Item A: $U_A = \{User 1, User 2, User 3, User 4\}$
2.  Identify users who rated Item D: $U_D = \{User 1, User 2, User 3\}$
3.  Calculate Intersection size: $|U_A \cap U_D| = 3$
4.  Calculate Union size: $|U_A \cup U_D| = 4$
5.  Calculate Jaccard Similarity:
    $$Jaccard(A, D) = \frac{|U_A \cap U_D|}{|U_A \cup U_D|} = \frac{3}{4} = 0.75$$

---

**Task 2 (Cosine):**
Calculate the Cosine similarity between **Item A** and **Item D**. Use the rating vectors for the users who rated both items.

**Solution:**
1.  Define Rating Vector for A: $\vec{A} = [5, 4, 1]$
2.  Define Rating Vector for D: $\vec{D} = [1, 2, 5]$
3.  Calculate Dot Product:
    $$\vec{A} \cdot \vec{D} = (5 \times 1) + (4 \times 2) + (1 \times 5) = 5 + 8 + 5 = 18$$
4.  Calculate Euclidean Norm of A:
    $$||\vec{A}|| = \sqrt{5^2 + 4^2 + 1^2} = \sqrt{25 + 16 + 1} = \sqrt{42} \approx 6.48$$
5.  Calculate Euclidean Norm of D:
    $$||\vec{D}|| = \sqrt{1^2 + 2^2 + 5^2} = \sqrt{1 + 4 + 25} = \sqrt{30} \approx 5.48$$
6.  Calculate Cosine Similarity:
    $$Cosine(A, D) = \frac{\vec{A} \cdot \vec{D}}{||\vec{A}|| \times ||\vec{D}||} = \frac{18}{\sqrt{42} \times \sqrt{30}} = \frac{18}{\sqrt{1260}} \approx 0.507$$

---

**Task 3 (Prediction):**
Predict **User 2's rating for Item B** using item-based collaborative filtering. Take k = 2 and use Jaccard sim.
*   Assume $Jaccard(B, A) = 0.75$
*   Assume $Jaccard(B, D) = 0.5$
*   User 2 ratings: Item A (4), Item D (2)

**Solution:**
1.  Identify neighbors and weights:
    *   Neighbor 1: Item A, Rating $r_{u2,A} = 4$, Weight $w_A = 0.75$
    *   Neighbor 2: Item D, Rating $r_{u2,D} = 2$, Weight $w_D = 0.5$
2.  Calculate Weighted Sum (Numerator):
    $$\sum (sim \times rating) = (0.75 \times 4) + (0.5 \times 2) = 3.0 + 1.0 = 4.0$$
3.  Calculate Sum of Weights (Denominator):
    $$\sum sim = 0.75 + 0.5 = 1.25$$
4.  Calculate Final Prediction:
    $$r_{u2, B} = \frac{4.0}{1.25} = 3.2$$
